# SmolVLA in the scaled LIBERO scene: benchmark of pretrained checkpoints

**Question.** Before training on our own generated data, how do the most popular SmolVLA LIBERO checkpoints behave (a) in LIBERO's own scene, (b) with the table cleared, (c) with only the bowl and plate on the table, at the measured real sizes (the target scene of experiment E1), and (d) when the bowl and plate start exactly where LIBERO's init state puts them, so only the scene clutter and object size differ from stock?

**Two kinds of test.**

| Test | What it does | Where |
|---|---|---|
| **Closed loop** | The policy drives the Panda in the simulator; success is LIBERO's own check. `replan` = how many actions of a predicted chunk run before the policy looks again (10 = look every 0.5 s, 50 = one look per 2.5 s chunk, close to open loop) | `hurobo.policy_rollout`, `hurobo.benchmark` |
| **Open loop** | Predicted action chunks are compared with demonstrated ones on recorded frames; no simulator | `hurobo.benchmark.OpenLoopEvaluator` |

Conditions: `stock` (LIBERO untouched), `clean` (other objects removed, LIBERO sizes, our layouts), `scaled` (clean + real sizes), plus `clean_stock_layout` / `scaled_stock_layout` (the same two scenes with LIBERO's own bowl and plate placement). Same layouts for every checkpoint (layout seed = episode number).

Everything below is read from `eval/pretrained_libero_checkpoints/` (written by `scripts/18_benchmark_policy.py`); the commands are at the end.

In [1]:
from hurobo.utils.notebook import setup_notebook

ROOT = setup_notebook()

In [2]:
import json

import pandas as pd

from hurobo.benchmark import OPEN_LOOP_FILE, load_episode_rows, summarize_rows
from hurobo.policy_rollout import CONDITION_HELP, KNOWN_CHECKPOINTS
from hurobo.utils.viewer import benchmark_success_figure, failure_stage_figure, open_loop_figure

RESULTS = ROOT / "eval/pretrained_libero_checkpoints"
rows = load_episode_rows(RESULTS)
summary = summarize_rows(rows)
print(f"{len(rows)} episodes stored")
pd.DataFrame(CONDITION_HELP, index=["meaning"]).T

/home/eeschner/alfredo/workspace/hurobo/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


103 episodes stored


,meaning
stock,"LIBERO's own scene and init states, untouched ..."
clean,"LIBERO's bowl/plate sizes, other objects remov..."
scaled,"bowl and plate at the measured real sizes, oth..."
clean_stock_layout,"as clean, but bowl and plate start where LIBER..."
scaled_stock_layout,"as scaled, but bowl and plate start where LIBE..."


In [3]:
pd.DataFrame(
    [
        {"name": s.name, "Hub likes": s.likes, "note": s.note, "needs camera rename": bool(s.rename_map)}
        for s in KNOWN_CHECKPOINTS.values()
    ]
)

,name,Hub likes,note,needs camera rename
0,HuggingFaceVLA/smolvla_libero,20,41.7k downloads/month; image/image2 keys; trai...,False
1,lerobot/smolvla_libero,7,"trained on lerobot/libero (all four suites), 2...",True


## 1. Closed-loop success

Bars are the success rate, whiskers the 95 % Wilson interval (few episodes per cell, so they are wide).

In [4]:
benchmark_success_figure(summary).show()
summary.round(2)

,checkpoint,condition,replan_every,episodes,successes,success_rate,ci_low,ci_high,lifted,mean_steps_success,median_min_eef_bowl_cm
0,HuggingFaceVLA/smolvla_libero,clean,10,10,0,0.0,0.00,0.28,0.0,NaN,13.65
1,HuggingFaceVLA/smolvla_libero,clean_stock_layout,10,10,3,0.3,0.11,0.60,0.3,103.00,7.56
2,HuggingFaceVLA/smolvla_libero,scaled,10,10,0,0.0,0.00,0.28,0.0,NaN,14.62
3,HuggingFaceVLA/smolvla_libero,scaled_stock_layout,10,8,0,0.0,0.00,0.32,0.0,NaN,10.46
4,HuggingFaceVLA/smolvla_libero,stock,10,10,8,0.8,0.49,0.94,0.9,105.00,5.54
5,lerobot/smolvla_libero,clean,10,10,0,0.0,0.00,0.28,0.2,NaN,15.62
6,lerobot/smolvla_libero,clean_stock_layout,10,10,6,0.6,0.31,0.83,1.0,80.67,4.59
7,lerobot/smolvla_libero,scaled,10,10,0,0.0,0.00,0.28,0.2,NaN,14.61
8,lerobot/smolvla_libero,scaled,50,5,0,0.0,0.00,0.43,0.4,NaN,13.47
9,lerobot/smolvla_libero,scaled_stock_layout,10,10,0,0.0,0.00,0.28,0.4,NaN,6.76


## 2. How far did the episodes get?

`lifted` is the share of episodes in which the bowl rose more than 3 cm, so a low success with a high `lifted` means the grasp works but the placement does not, and a low `lifted` means the policy never picked the bowl up.

In [5]:
failure_stage_figure(summary).show()

## 2b. Same object placement as stock

`clean_stock_layout` and `scaled_stock_layout` keep LIBERO's init-state placement of the bowl and plate and only change the rest of the scene: if these succeed where `clean` / `scaled` (random layouts) fail, the policies depend on the placement they saw in training, not on the clutter. In `scaled_stock_layout` the real-size plate may overlap the bowl, because LIBERO's spacing was made for the small plate.

In [6]:
layout_conditions = ["stock", "clean", "clean_stock_layout", "scaled", "scaled_stock_layout"]
summary[(summary.replan_every == 10) & summary.condition.isin(layout_conditions)].pivot(
    index="checkpoint", columns="condition", values="success_rate"
)[layout_conditions].round(2)

condition,stock,clean,clean_stock_layout,scaled,scaled_stock_layout
checkpoint,,,,,
HuggingFaceVLA/smolvla_libero,0.8,0.0,0.3,0.0,0.0
lerobot/smolvla_libero,1.0,0.0,0.6,0.0,0.0


## 3. Open-loop error against demonstrations

Demonstrations are the generated robot demos (stand-in hand paths until real clips exist). A dimension where the policy's error is not below the zero-action bar tells us nothing about that dimension.

In [7]:
open_loop_path = RESULTS / OPEN_LOOP_FILE
if open_loop_path.exists():
    open_loop_figure(json.loads(open_loop_path.read_text())).show()
else:
    print("no open-loop results yet: run scripts/18_benchmark_policy.py with --open-loop-dataset")

## 4. Rollout videos (first episode of each cell)

In [8]:
from hurobo.utils.notebook import show_video

for video in sorted((RESULTS / "videos").glob("*ep0.mp4")):
    print(video.stem)
    show_video(video, width=256)

HuggingFaceVLA_smolvla_libero__clean__10__ep0
HuggingFaceVLA_smolvla_libero__clean_stock_layout__10__ep0
HuggingFaceVLA_smolvla_libero__scaled__10__ep0
HuggingFaceVLA_smolvla_libero__scaled_stock_layout__10__ep0
HuggingFaceVLA_smolvla_libero__stock__10__ep0
lerobot_smolvla_libero__clean__10__ep0
lerobot_smolvla_libero__clean_stock_layout__10__ep0
lerobot_smolvla_libero__scaled__10__ep0
lerobot_smolvla_libero__scaled__50__ep0
lerobot_smolvla_libero__scaled_stock_layout__10__ep0
lerobot_smolvla_libero__stock__10__ep0


## Commands

```bash
# 1. closed loop, the two most-liked checkpoints, three conditions, two replan intervals
MUJOCO_GL=egl uv run python scripts/18_benchmark_policy.py --checkpoints lerobot hfvla \
    --conditions stock clean scaled --replan 10 50 --episodes 10 --out eval/pretrained_libero_checkpoints
# 2. open loop against a generated dataset
MUJOCO_GL=egl uv run python scripts/18_benchmark_policy.py --checkpoints lerobot hfvla --open-loop-only \
    --open-loop-dataset data/e1/standin_generated --out eval/pretrained_libero_checkpoints
# 3. train on generated data, then benchmark the result the same way
uv run python scripts/19_train_smolvla.py data/e1/generated --out outputs/e1_smolvla
MUJOCO_GL=egl uv run python scripts/18_benchmark_policy.py \
    --checkpoint-path e1=outputs/e1_smolvla/checkpoints/last/pretrained_model --conditions scaled
```